In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parents[1]
sys.path.append(str(PROJECT_ROOT))

In [26]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from loaders._load_vn30_meta import _process_file, VN30, TARGETS
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import TimeSeriesSplit, RandomizedSearchCV
from sklearn.metrics import r2_score, mean_absolute_percentage_error, root_mean_squared_error

In [7]:
def preprocess(
    symbol: str,
    lag: int = 30,
    val: float = 0.0,
    verbose: bool = False,
):
    """
    Tiền xử lý dùng sai phân cho mô hình cây (không scale):
      - Feature X: các cột diff_lag_{1..lag} của 4 giá trị TARGETS trong {lag} ngày trước.
      - Target y: diff của ngày t+1 so với t (dịch -1).
      - Trả về kèm Y_base (giá trị gốc tại t) và Y_true (giá trị gốc tại t+1) để cộng ngược diff sau dự đoán.
      - Chia test theo đúng chiều dài df_test, đảm bảo số mẫu test == len(df_test).
    """
    # 1) Đọc & nối train/test, bỏ volume
    df_train, df_test = _process_file(symbol)
    df_train = df_train.drop(columns=['volume'])
    df_test  = df_test.drop(columns=['volume'])
    test_size = len(df_test)

    df_all = pd.concat([df_train, df_test], ignore_index=True)

    # 2) Sai phân cho từng cột mục tiêu
    for feat in TARGETS:
        df_all[f"{feat}_diff"] = df_all[feat].diff()

    # 3) Tạo X: lag trên *_diff
    diff_lag_cols = {}
    for feat in TARGETS:
        for i in range(1, lag + 1):
            diff_lag_cols[f"{feat}_diff_lag_{i}"] = df_all[f"{feat}_diff"].shift(i)
    X_block = pd.DataFrame(diff_lag_cols, index=df_all.index)

    # 4) Tạo y: diff của ngày t+1 (dịch -1)
    y_block = df_all[[f"{feat}_diff" for feat in TARGETS]].shift(-1)
    y_block.columns = [f"{feat}_y" for feat in TARGETS]  # open_y, high_y, low_y, close_y

    # 5) Tạo Y_base (giá trị gốc ở ngày t) và Y_true (giá trị gốc ở ngày t+1)
    Y_base_block = df_all[TARGETS].add_suffix('_base')          # ..._base = giá trị tại t
    Y_true_block = df_all[TARGETS].shift(-1).add_suffix('_true')# ..._true = giá trị tại t+1

    # 6) Ghép khung rồi dropna một lần
    frames = []
    if 'time' in df_all.columns:
        frames.append(df_all[['time']])  # chỉ để giữ index đồng bộ, sẽ drop sau
    frames.extend([X_block, y_block, Y_base_block, Y_true_block])
    work = pd.concat(frames, axis=1)

    if 'time' in work.columns:
        work = work.drop(columns=['time'])
    work = work.dropna()

    # 7) Tách X / y / Y_base / Y_true
    y_cols      = [f"{feat}_y" for feat in TARGETS]
    y_base_cols = [f"{feat}_base" for feat in TARGETS]
    y_true_cols = [f"{feat}_true" for feat in TARGETS]

    X = work.drop(columns=y_cols + y_base_cols + y_true_cols).values
    y = work[y_cols].values
    Y_base_all = work[y_base_cols].values   # (n_all, 4)
    Y_true_all = work[y_true_cols].values   # (n_all, 4)

    # 8) Chia train/test theo test_size (dòng cuối tương ứng các ngày test)
    X_train_full, X_test = X[:-test_size], X[-test_size:]
    y_train_full, y_test = y[:-test_size], y[-test_size:]
    Y_base_train_full, Y_base_test = Y_base_all[:-test_size], Y_base_all[-test_size:]
    Y_true_train_full, Y_true_test = Y_true_all[:-test_size], Y_true_all[-test_size:]

    # 9) Chia val từ phần train (nếu cần)
    n_samples = X_train_full.shape[0]
    valid_size = int(n_samples * val)
    train_size = n_samples - valid_size

    X_train, X_val   = X_train_full[:train_size], X_train_full[train_size:]
    y_train, y_val   = y_train_full[:train_size], y_train_full[train_size:]
    Y_base_train, Y_base_val = Y_base_train_full[:train_size], Y_base_train_full[train_size:]
    Y_true_train, Y_true_val = Y_true_train_full[:train_size], Y_true_train_full[train_size:]

    if verbose:
        print(f"=== Preprocessing (diff) {symbol} ===")
        print(f"Shapes X: train {X_train.shape}, val {X_val.shape}, test {X_test.shape}")
        print(f"Shapes y(diff): train {y_train.shape}, val {y_val.shape}, test {y_test.shape}")
        # Kiểm tra cột:
        assert y_train.shape[1] == len(TARGETS) == 4
        assert Y_true_test.shape[1] == len(TARGETS) == 4

    return {
        "train": (X_train, y_train),
        "val":   (X_val,   y_val),
        "test":  (X_test,  y_test),
        # Trả về luôn Y_base & Y_true (giá trị gốc) để cộng ngược diff sau khi dự đoán
        "Y_base": {"train": Y_base_train, "val": Y_base_val, "test": Y_base_test},
        "Y_true": {"train": Y_true_train, "val": Y_true_val, "test": Y_true_test},
    }

In [8]:
_ = preprocess('ACB', lag=30, verbose=True)

=== Preprocessing (diff) ACB ===
Shapes X: train (1213, 120), val (0, 120), test (328, 120)
Shapes y(diff): train (1213, 4), val (0, 4), test (328, 4)


In [14]:
tracks = {"r2": [], "mape": []}
for symbol in VN30:
    data = preprocess(symbol, lag=2)
    X_train, Y_train = data["train"]
    X_test, Y_test = data["test"]
    Y_base_test = data["Y_base"]["test"]
    Y_true_test = data["Y_true"]["test"]

    tscv = TimeSeriesSplit(n_splits=3)
    model = RandomizedSearchCV(
        estimator=DecisionTreeRegressor(),
        param_distributions={
            "max_depth": [3, 5, 7, 9],
            "min_samples_split": [2, 5, 10],
            "min_samples_leaf": [1, 2, 4]
        }, 
        cv=tscv, 
        n_iter=10, 
        random_state=42
    )

    model.fit(X_train, Y_train)
    Y_pred_diff = model.predict(X_test)
    Y_pred = Y_base_test + Y_pred_diff

    r2 = r2_score(Y_true_test, Y_pred)
    mape = mean_absolute_percentage_error(Y_true_test, Y_pred) * 100

    tracks["r2"].append(r2)
    tracks["mape"].append(mape)

    print(f"Symbol: {symbol}, R2: {r2:.4f}, MAPE: {mape:.4f}")

print(f"Mean R2: {np.mean(tracks['r2']):.4f}, Mean MAPE: {np.mean(tracks['mape']):.4f}")
print(f"Std R2: {np.std(tracks['r2']):.4f}, Std MAPE: {np.std(tracks['mape']):.4f}")

Symbol: ACB, R2: 0.9369, MAPE: 0.8859
Symbol: BCM, R2: 0.9479, MAPE: 1.4145
Symbol: BID, R2: 0.9024, MAPE: 1.1001
Symbol: BVH, R2: 0.9735, MAPE: 1.1637
Symbol: CTG, R2: 0.9628, MAPE: 1.1706
Symbol: FPT, R2: 0.9886, MAPE: 1.2298
Symbol: GAS, R2: 0.9491, MAPE: 0.8062
Symbol: GVR, R2: 0.9611, MAPE: 1.8036
Symbol: HDB, R2: 0.9712, MAPE: 1.1598
Symbol: HPG, R2: 0.8997, MAPE: 1.0458
Symbol: LPB, R2: 0.9942, MAPE: 1.3946
Symbol: MBB, R2: 0.9494, MAPE: 1.1453
Symbol: MSN, R2: 0.9466, MAPE: 1.2320
Symbol: MWG, R2: 0.9798, MAPE: 1.3056
Symbol: PLX, R2: 0.9766, MAPE: 1.2008
Symbol: SAB, R2: 0.9138, MAPE: 0.9752
Symbol: SHB, R2: 0.9513, MAPE: 1.0269
Symbol: SSB, R2: 0.9514, MAPE: 0.9680
Symbol: SSI, R2: 0.9233, MAPE: 1.1820
Symbol: STB, R2: 0.9739, MAPE: 1.2023
Symbol: TCB, R2: 0.9767, MAPE: 1.1916
Symbol: TPB, R2: 0.9433, MAPE: 1.1837
Symbol: VCB, R2: 0.8456, MAPE: 0.7921
Symbol: VHM, R2: 0.9600, MAPE: 1.2304
Symbol: VIB, R2: 0.9244, MAPE: 0.9650
Symbol: VIC, R2: 0.9631, MAPE: 1.1758
Symbol: VJC,

In [15]:
tracks = {"r2": [], "mape": []}
for symbol in VN30:
    data = preprocess(symbol, lag=2)
    X_train, Y_train = data["train"]
    X_test, Y_test = data["test"]
    Y_base_test = data["Y_base"]["test"]
    Y_true_test = data["Y_true"]["test"]

    tscv = TimeSeriesSplit(n_splits=3)
    model = RandomizedSearchCV(
        estimator=RandomForestRegressor(n_jobs=-1, random_state=42),
        param_distributions={
            "n_estimators": [10, 20, 50],
            "max_depth": [3, 5, 7, 9],
            "min_samples_split": [2, 5, 10],
            "min_samples_leaf": [1, 2, 4]
        }, 
        cv=tscv, 
        n_iter=10, 
        random_state=42
    )

    model.fit(X_train, Y_train)
    Y_pred_diff = model.predict(X_test)
    Y_pred = Y_base_test + Y_pred_diff

    r2 = r2_score(Y_true_test, Y_pred)
    mape = mean_absolute_percentage_error(Y_true_test, Y_pred) * 100

    tracks["r2"].append(r2)
    tracks["mape"].append(mape)

    print(f"Symbol: {symbol}, R2: {r2:.4f}, MAPE: {mape:.4f}")

print(f"Mean R2: {np.mean(tracks['r2']):.4f}, Mean MAPE: {np.mean(tracks['mape']):.4f}")
print(f"Std R2: {np.std(tracks['r2']):.4f}, Std MAPE: {np.std(tracks['mape']):.4f}")

Symbol: ACB, R2: 0.9365, MAPE: 0.8668
Symbol: BCM, R2: 0.9468, MAPE: 1.4214
Symbol: BID, R2: 0.9021, MAPE: 1.0915
Symbol: BVH, R2: 0.9732, MAPE: 1.1715
Symbol: CTG, R2: 0.9631, MAPE: 1.1671
Symbol: FPT, R2: 0.9889, MAPE: 1.2063
Symbol: GAS, R2: 0.9497, MAPE: 0.8121
Symbol: GVR, R2: 0.9637, MAPE: 1.7811
Symbol: HDB, R2: 0.9725, MAPE: 1.1420
Symbol: HPG, R2: 0.9034, MAPE: 1.0279
Symbol: LPB, R2: 0.9943, MAPE: 1.3805
Symbol: MBB, R2: 0.9498, MAPE: 1.1253
Symbol: MSN, R2: 0.9472, MAPE: 1.2195
Symbol: MWG, R2: 0.9802, MAPE: 1.3058
Symbol: PLX, R2: 0.9779, MAPE: 1.1793
Symbol: SAB, R2: 0.9215, MAPE: 0.9460
Symbol: SHB, R2: 0.9549, MAPE: 0.9952
Symbol: SSB, R2: 0.9517, MAPE: 0.9592
Symbol: SSI, R2: 0.9293, MAPE: 1.1547
Symbol: STB, R2: 0.9749, MAPE: 1.1726
Symbol: TCB, R2: 0.9768, MAPE: 1.1721
Symbol: TPB, R2: 0.9445, MAPE: 1.1733
Symbol: VCB, R2: 0.8546, MAPE: 0.7855
Symbol: VHM, R2: 0.9590, MAPE: 1.2263
Symbol: VIB, R2: 0.9230, MAPE: 0.9680
Symbol: VIC, R2: 0.9634, MAPE: 1.1770
Symbol: VJC,

# Diversity-Driven Forest

In [27]:
# ===== 1) Huấn luyện 1 cây với subspace & random search =====
def fit_one_tree(X_tr, y_tr, feature_idx, n_splits=3, random_state=42):
    tscv = TimeSeriesSplit(n_splits=n_splits)
    est = DecisionTreeRegressor(random_state=random_state)
    search = RandomizedSearchCV(
        estimator=est,
        param_distributions={
            "max_depth": [3,5,7,9],
            "min_samples_split": [2,5,10],
            "min_samples_leaf": [1,2,4],
        },
        cv=tscv, n_iter=10, random_state=random_state
    )
    search.fit(X_tr[:, feature_idx], y_tr)
    return search.best_estimator_, feature_idx

# ===== 2) Xây pool cây =====
def build_pool(X_tr, y_tr, X_val, M=64, mtry=None, seed=42):
    rng = np.random.default_rng(seed)
    d = X_tr.shape[1]
    mtry = mtry or max(1, int(np.sqrt(d)))
    pool = []
    for m in range(M):
        feat_idx = rng.choice(d, size=mtry, replace=False)
        tree, feat_idx = fit_one_tree(X_tr, y_tr, feat_idx, random_state=seed+m)
        yhat_val = tree.predict(X_val[:, feat_idx])                 # (n_val, 4)
        pool.append({"tree": tree, "feat": feat_idx, "yhat_val": yhat_val})
    return pool

# ===== 3) Greedy chọn tập con đa dạng (unweighted) =====
def _vec_center(Y):
    v = Y.reshape(-1)
    return v - v.mean()

def _safe_corr(a, b, eps=1e-12):
    sa, sb = a.std(), b.std()
    if sa < eps or sb < eps: return 0.0
    return float(np.corrcoef(a, b)[0,1])

def select_diverse_unweighted(pool, Y_val, k=16, beta=0.5):
    n = len(pool)
    H = [_vec_center(p["yhat_val"]) for p in pool]
    y = _vec_center(Y_val)

    # start: best RMSE single
    rmses = [root_mean_squared_error(Y_val, pool[i]["yhat_val"]) for i in range(n) ]
    S = [int(np.argmin(rmses))]

    while len(S) < min(k, n):
        best_c, best_obj = None, None
        # current average
        avg = sum(pool[j]["yhat_val"] for j in S) / len(S)
        for c in range(n):
            if c in S: continue
            avg_c = (avg*len(S) + pool[c]["yhat_val"]) / (len(S)+1)
            rmse_c = root_mean_squared_error(Y_val, avg_c)
            # mean |corr| with selected
            corr_c = np.mean([abs(_safe_corr(H[c], H[j])) for j in S]) if S else 0.0
            obj = rmse_c + beta * corr_c
            if (best_obj is None) or (obj < best_obj):
                best_obj, best_c = obj, c
        S.append(best_c)

    return S  # indices in pool

# ===== 4) Tính trọng số (tùy chọn) =====
def stack_weights_with_diversity(pool, S, Y_val, lam=0.1, nonneg=True, sum_to_one=True):
    # Build H_S (centered) & y (centered)
    H_cols = [ _vec_center(pool[i]["yhat_val"]) for i in S ]
    H = np.column_stack(H_cols)                 # (n_val*d, k)
    y = _vec_center(Y_val)                      # (n_val*d,)
    # correlation matrix C_S
    k = len(S)
    C = np.eye(k)
    for i in range(k):
        for j in range(i+1, k):
            C[i,j] = C[j,i] = _safe_corr(H[:,i], H[:,j])

    # ridge-like closed form, then project
    A = H.T @ H + lam * C
    b = H.T @ y
    w = np.linalg.solve(A + 1e-8*np.eye(k), b)

    # optional constraints
    if nonneg:
        w = np.maximum(w, 0.0)
    if sum_to_one:
        s = w.sum()
        w = w / s if s > 1e-12 else np.ones_like(w)/k
    return w  # shape (k,)

# ===== 5) Suy luận trên test =====
def predict_ensemble(pool, S, X_test, Y_base_test, weights=None):
    yhats = []
    for i in S:
        tree, feat = pool[i]["tree"], pool[i]["feat"]
        yhats.append(tree.predict(X_test[:, feat]))           # (n_test, 4)
    yhats = np.stack(yhats, axis=0)                           # (k, n_test, 4)
    if weights is None:
        y_pred_diff = yhats.mean(axis=0)
    else:
        w = np.asarray(weights).reshape(-1,1,1)               # (k,1,1)
        y_pred_diff = (w * yhats).sum(axis=0)
    return Y_base_test + y_pred_diff

In [28]:
tracks = {"r2": [], "mape": []}
for symbol in VN30:
    # 0) Lấy dữ liệu
    data = preprocess(symbol, lag=30, val=0.2)
    X_train, y_train = data["train"]
    X_val,   y_val   = data["val"]
    X_test,  y_test  = data["test"]  # (diff) — chỉ dùng để tham khảo
    Y_base_test = data["Y_base"]["test"]
    Y_true_test = data["Y_true"]["test"]

    # 1) Xây pool (M cây)
    pool = build_pool(X_train, y_train, X_val, M=64, mtry=None, seed=42)

    # 2) Chọn tập con đa dạng (k cây) – unweighted
    S = select_diverse_unweighted(pool, y_val, k=16, beta=0.5)

    # 3a) Không trọng số
    Y_pred = predict_ensemble(pool, S, X_test, Y_base_test, weights=None)

    # 3b) (tuỳ chọn) Có trọng số với phạt đa dạng
    w = stack_weights_with_diversity(pool, S, y_val, lam=0.1, nonneg=True, sum_to_one=True)
    Y_pred = predict_ensemble(pool, S, X_test, Y_base_test, weights=w)

    # 4) Đánh giá toàn cục (multioutput)
    r2   = r2_score(Y_true_test, Y_pred)
    mape = mean_absolute_percentage_error(Y_true_test, Y_pred) * 100
    print(f"Symbol: {symbol}, R2: {r2:.4f}, MAPE: {mape:.4f}")
    tracks["r2"].append(r2)
    tracks["mape"].append(mape)

print(f"Mean R2: {np.mean(tracks['r2']):.4f}, Mean MAPE: {np.mean(tracks['mape']):.4f}")
print(f"Std R2: {np.std(tracks['r2']):.4f}, Std MAPE: {np.std(tracks['mape']):.4f}")

Symbol: ACB, R2: 0.9364, MAPE: 0.8743
Symbol: BCM, R2: 0.9480, MAPE: 1.4131
Symbol: BID, R2: 0.8988, MAPE: 1.1088
Symbol: BVH, R2: 0.9733, MAPE: 1.1716
Symbol: CTG, R2: 0.9640, MAPE: 1.1471
Symbol: FPT, R2: 0.9888, MAPE: 1.2104
Symbol: GAS, R2: 0.9511, MAPE: 0.8026
Symbol: GVR, R2: 0.9633, MAPE: 1.7998
Symbol: HDB, R2: 0.9727, MAPE: 1.1383
Symbol: HPG, R2: 0.9050, MAPE: 1.0180
Symbol: LPB, R2: 0.9943, MAPE: 1.3759
Symbol: MBB, R2: 0.9490, MAPE: 1.1418
Symbol: MSN, R2: 0.9466, MAPE: 1.2243
Symbol: MWG, R2: 0.9807, MAPE: 1.2879
Symbol: PLX, R2: 0.9778, MAPE: 1.1807
Symbol: SAB, R2: 0.9223, MAPE: 0.9396
Symbol: SHB, R2: 0.9560, MAPE: 0.9931
Symbol: SSB, R2: 0.9530, MAPE: 0.9457
Symbol: SSI, R2: 0.9287, MAPE: 1.1536
Symbol: STB, R2: 0.9749, MAPE: 1.1718
Symbol: TCB, R2: 0.9764, MAPE: 1.1627
Symbol: TPB, R2: 0.9443, MAPE: 1.1732
Symbol: VCB, R2: 0.8615, MAPE: 0.7792
Symbol: VHM, R2: 0.9585, MAPE: 1.2319
Symbol: VIB, R2: 0.9262, MAPE: 0.9613
Symbol: VIC, R2: 0.9631, MAPE: 1.1702
Symbol: VJC,

In [30]:
from xgboost import XGBRegressor

tracks = {"r2": [], "mape": []}
for symbol in VN30:
    data = preprocess(symbol, lag=2)
    X_train, Y_train = data["train"]
    X_test, Y_test = data["test"]
    Y_base_test = data["Y_base"]["test"]
    Y_true_test = data["Y_true"]["test"]

    tscv = TimeSeriesSplit(n_splits=3)
    model = RandomizedSearchCV(
        estimator=XGBRegressor(n_jobs=-1, random_state=42),
        param_distributions={
            "n_estimators": [10, 20, 50],
            "max_depth": [3, 5, 7, 9]
        }, 
        cv=tscv, 
        n_iter=10, 
        random_state=42
    )

    model.fit(X_train, Y_train)
    Y_pred_diff = model.predict(X_test)
    Y_pred = Y_base_test + Y_pred_diff

    r2 = r2_score(Y_true_test, Y_pred)
    mape = mean_absolute_percentage_error(Y_true_test, Y_pred) * 100

    tracks["r2"].append(r2)
    tracks["mape"].append(mape)

    print(f"Symbol: {symbol}, R2: {r2:.4f}, MAPE: {mape:.4f}")

print(f"Mean R2: {np.mean(tracks['r2']):.4f}, Mean MAPE: {np.mean(tracks['mape']):.4f}")
print(f"Std R2: {np.std(tracks['r2']):.4f}, Std MAPE: {np.std(tracks['mape']):.4f}")

Symbol: ACB, R2: 0.9316, MAPE: 0.8884
Symbol: BCM, R2: 0.9453, MAPE: 1.4253
Symbol: BID, R2: 0.9010, MAPE: 1.1026
Symbol: BVH, R2: 0.9719, MAPE: 1.2010
Symbol: CTG, R2: 0.9632, MAPE: 1.1690
Symbol: FPT, R2: 0.9884, MAPE: 1.2314
Symbol: GAS, R2: 0.9474, MAPE: 0.8294
Symbol: GVR, R2: 0.9617, MAPE: 1.8307
Symbol: HDB, R2: 0.9718, MAPE: 1.1663
Symbol: HPG, R2: 0.9007, MAPE: 1.0532
Symbol: LPB, R2: 0.9940, MAPE: 1.4094
Symbol: MBB, R2: 0.9495, MAPE: 1.1352
Symbol: MSN, R2: 0.9453, MAPE: 1.2478
Symbol: MWG, R2: 0.9794, MAPE: 1.3382
Symbol: PLX, R2: 0.9769, MAPE: 1.1950
Symbol: SAB, R2: 0.9161, MAPE: 0.9753
Symbol: SHB, R2: 0.9559, MAPE: 0.9909
Symbol: SSB, R2: 0.9503, MAPE: 0.9851
Symbol: SSI, R2: 0.9279, MAPE: 1.1715
Symbol: STB, R2: 0.9752, MAPE: 1.1838
Symbol: TCB, R2: 0.9769, MAPE: 1.1820
Symbol: TPB, R2: 0.9416, MAPE: 1.2022
Symbol: VCB, R2: 0.8499, MAPE: 0.7940
Symbol: VHM, R2: 0.9568, MAPE: 1.2478
Symbol: VIB, R2: 0.9191, MAPE: 0.9973
Symbol: VIC, R2: 0.9628, MAPE: 1.1949
Symbol: VJC,